# PromptSec-FM XLM-R Multitask Training

This workflow trains on synthetic, automatically validated **SILVER** labels. They are not human Gold truth, and good PolicyBench scores do not prove real-world prompt-injection robustness. Full training requires a CUDA GPU.

## 1. Project overview and SILVER warning

Nine frozen taxonomy heads share one multilingual XLM-R encoder. Span prediction is intentionally deferred to a future phase.

## 2. Google Drive mounting

Authorize only the Drive account that owns the configured PromptSec-FM directory.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. User configuration

Edit this one cell before running the notebook. No private Drive identifier or credential is hardcoded.

In [ ]:
PROJECT_NAME = "PromptSec-FM"
RUN_NAME = "xlmr-base-multitask-v0.1"
MODEL_NAME = "FacebookAI/xlm-roberta-base"
REPO_URL = ""  # Optional public/fork Git URL; leave empty if REPO_DIR already exists.
REPO_DIR = "/content/PromptSec-FM"
DRIVE_ROOT = "/content/drive/MyDrive/PromptSec-FM"
DATA_ARCHIVE = f"{DRIVE_ROOT}/data/policybench-codex-v0.1.zip"
DATA_SHA256_FILE = f"{DATA_ARCHIVE}.sha256"
DATA_ROOT = "/content/promptsec_data/policybench-codex-v0.1"
CHECKPOINT_ROOT = f"{DRIVE_ROOT}/checkpoints/{RUN_NAME}"
REPORT_ROOT = f"{DRIVE_ROOT}/reports/{RUN_NAME}"
TRAINING_MODE = "SCIENTIFIC_EVALUATION"
MAX_LENGTH = 512
NUM_EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
EARLY_STOPPING_PATIENCE = 2
SEED = 20260718
RESUME = True
RUN_SMOKE_TEST_FIRST = True
RUN_FULL_TRAINING = True
ENABLE_HF_LOGIN = False

## 4. GPU and runtime preflight

The full run aborts rather than silently using CPU. A CPU smoke test is permitted only through the explicit smoke flag.

In [ ]:
import importlib.metadata
import os
import platform
import shutil
from pathlib import Path

import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
try:
    print("Transformers:", importlib.metadata.version("transformers"))
except importlib.metadata.PackageNotFoundError:
    print("Transformers: not installed yet")
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total/free GiB:", total / 2**30, free / 2**30)
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    print("Selected precision:", "bf16" if torch.cuda.is_bf16_supported() else "fp16")
else:
    print("GPU: none; full training will abort, explicit smoke test may use CPU")
page_size = os.sysconf("SC_PAGE_SIZE")
pages = os.sysconf("SC_PHYS_PAGES")
available_pages = os.sysconf("SC_AVPHYS_PAGES")
print(
    "System RAM total/available GiB:",
    page_size * pages / 2**30,
    page_size * available_pages / 2**30,
)
if Path(DRIVE_ROOT).exists():
    print("Drive free GiB:", shutil.disk_usage(DRIVE_ROOT).free / 2**30)

## 5. Repository setup

Clone only when the repository is not already present in the Colab runtime.

In [ ]:
import os
import subprocess
from pathlib import Path

if not Path(REPO_DIR).is_dir():
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL or upload/mount the repository at REPO_DIR.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Repository:", Path.cwd())

## 6. Dependency installation

Install the repository's bounded training extra; no experiment-tracking service or API key is used.

In [ ]:
%pip install -q -e '.[training]'

## 7. Dataset archive verification

The uploaded ZIP is rejected before extraction when its SHA-256 differs.

In [ ]:
import hashlib
from pathlib import Path

archive = Path(DATA_ARCHIVE)
sha_file = Path(DATA_SHA256_FILE)
if not archive.is_file() or not sha_file.is_file():
    raise FileNotFoundError("Upload both ZIP and .sha256 files to DRIVE_ROOT/data.")
expected = sha_file.read_text(encoding="utf-8").split()[0].lower()
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
if digest != expected:
    raise RuntimeError(f"Archive SHA-256 mismatch: {digest} != {expected}")
print("Verified archive SHA-256:", digest)

## 8. Dataset extraction

Training uses the local Colab disk, not the compressed Drive file.

In [ ]:
import shutil
import zipfile
from pathlib import Path

extract_root = Path("/content/promptsec_data")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(DATA_ARCHIVE) as zf:
    root = extract_root.resolve()
    for member in zf.infolist():
        target = (root / member.filename).resolve()
        if root not in target.parents and target != root:
            raise RuntimeError("Unsafe ZIP path")
    zf.extractall(root)
print("Extracted:", DATA_ROOT)

## 9. Release and split integrity checks

This performs packaged checksums, canonical state, exact split counts, ID uniqueness, and leakage gates before model loading.

In [ ]:
from promptsec.training.dataset import load_training_dataset

bundle = load_training_dataset(DATA_ROOT)
print(bundle.integrity_report)
assert bundle.integrity_report["automatic_gold_records"] == 0

## 10. Label-vocabulary inspection

Mappings come from the packaged frozen schema and are stored with hashes.

In [ ]:
for head, mapping in bundle.mappings.items():
    print(head, mapping.labels, mapping.mapping_hash)

## 11. Tokenization and serialization preview using redacted examples

Only section token lengths and a text hash are displayed; attack payloads are not printed.

In [ ]:
from transformers import AutoTokenizer

from promptsec.training.serialization import (
    SPECIAL_TOKENS,
    record_sections,
    serialize_full_context,
)

sample = bundle.records_by_split["train"][0]
serialized = serialize_full_context(sample)
preview_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
preview_tokenizer.add_special_tokens({"additional_special_tokens": list(SPECIAL_TOKENS)})
print("record_id:", sample["id"])
print("serialized_sha256:", hashlib.sha256(serialized.encode()).hexdigest())
section_token_lengths = {
    name: len(preview_tokenizer.encode(value, add_special_tokens=False))
    for name, value in record_sections(sample).as_ordered_items()
}
print("section_token_lengths:", section_token_lengths)

## 12. Optional small smoke test

Uses 32 training and 16 validation records, one epoch, length 128, separate checkpoints, backward pass, checkpoint reload, and one-step resume probe.

In [ ]:
import subprocess

base_cmd = [
    "python",
    "scripts/train_xlmr_multitask.py",
    "--config",
    "configs/xlmr_multitask_colab_v0.1.yaml",
    "--dataset",
    DATA_ROOT,
    "--output",
    CHECKPOINT_ROOT,
    "--reports",
    REPORT_ROOT,
    "--training-mode",
    TRAINING_MODE,
    "--model-name",
    MODEL_NAME,
    "--learning-rate",
    str(LEARNING_RATE),
    "--weight-decay",
    str(WEIGHT_DECAY),
    "--warmup-ratio",
    str(WARMUP_RATIO),
    "--early-stopping-patience",
    str(EARLY_STOPPING_PATIENCE),
    "--seed",
    str(SEED),
]
if RUN_SMOKE_TEST_FIRST:
    smoke_cmd = base_cmd + [
        "--smoke-test",
        "--max-train-records",
        "32",
        "--max-validation-records",
        "16",
        "--epochs",
        "1",
        "--max-length",
        "128",
        "--resume" if RESUME else "--no-resume",
    ]
    subprocess.run(smoke_cmd, check=True)

## 13. Full training

Runs the tested CLI. SCIENTIFIC_EVALUATION optimizes only the official train split and selects using validation only.

In [ ]:
from promptsec.training.checkpoints import checkpoint_inventory

before_training = checkpoint_inventory(CHECKPOINT_ROOT)
detected = [(item["path"], item["status"]) for item in before_training["checkpoints"]]
print("Detected checkpoints:", detected)
if RUN_FULL_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Colab GPU runtime before full training.")
    full_cmd = base_cmd + [
        "--epochs",
        str(NUM_EPOCHS),
        "--max-length",
        str(MAX_LENGTH),
        "--resume" if RESUME else "--no-resume",
    ]
    print("Training command:", " ".join(full_cmd))
    subprocess.run(full_cmd, check=True)

## 14. Automatic resume from Drive

The command above uses `--resume`. It verifies complete checkpoint checksums and fingerprints; incompatible state is rejected instead of silently restarting.

In [ ]:
from promptsec.training.checkpoints import checkpoint_inventory

inventory = checkpoint_inventory(CHECKPOINT_ROOT)
[(item["path"], item["status"]) for item in inventory["checkpoints"]]

## 15. Validation metrics

Validation controls early stopping, checkpoint selection, and optional multi-label threshold selection.

In [ ]:
import json

json.loads(Path(REPORT_ROOT, "validation_metrics.json").read_text())

## 16. Official test evaluation

The selected checkpoint is evaluated once on policy, domain, language, and counterfactual tests.

In [ ]:
json.loads(Path(REPORT_ROOT, "test_metrics.json").read_text())

## 17. Counterfactual evaluation

Includes pairwise, expected-change, invariant, exact-group, transition, and type-specific results.

In [ ]:
json.loads(Path(REPORT_ROOT, "counterfactual_results.json").read_text())

## 18. Language analysis

EN/FR differences are descriptive because the language-OOD split may differ in other ways.

In [ ]:
json.loads(Path(REPORT_ROOT, "language_results.json").read_text())

## 19. Hard-negative analysis

False positives are grouped by split, language, and category without printing payloads.

In [ ]:
json.loads(Path(REPORT_ROOT, "hard_negative_results.json").read_text())

## 20. Checkpoint export

The selected standalone model is under `best_model`. Hugging Face login is optional and disabled; no automatic upload occurs.

In [ ]:
best_model = Path(CHECKPOINT_ROOT, "best_model")
print("Best model:", best_model)
if ENABLE_HF_LOGIN:
    from huggingface_hub import login

    login()
print("No Hub upload is performed by this notebook.")

## 21. Model card generation

The CLI writes the model card locally on Drive with SILVER limitations.

In [ ]:
print(Path(REPORT_ROOT, "model_card.md").read_text(encoding="utf-8"))

## 22. Final run summary

Review integrity, effective settings, resource usage, resume events, and OOM recovery.

In [ ]:
print(Path(REPORT_ROOT, "final_report.md").read_text(encoding="utf-8"))

## 23. Reproduction instructions

Re-run the full command printed in section 13 with the same archive, configuration, and `--resume`. See `docs/xlmr_multitask_colab_v0.1.md` for PowerShell upload preparation and disconnect recovery.